In [ ]:
%pip install shap

In [ ]:
%pip uninstall -y shap numba llvmlite

In [ ]:
%pip install --no-cache-dir --force-reinstall "numpy==2.4.6" "numba==0.66.0" shap

In [ ]:
import sys
import importlib.metadata as metadata

print("Python executable:")
print(sys.executable)

packages = [
    "numpy",
    "numba",
    "llvmlite",
    "shap",
    "scikit-learn",
    "xgboost"
]

print("\nInstalled versions:")

for package in packages:
    try:
        print(
            f"{package}:",
            metadata.version(package)
        )
    except metadata.PackageNotFoundError:
        print(
            f"{package}: not installed"
        )

In [ ]:


import numpy as np
import numba
import shap

print("NumPy:", np.__version__)
print("Numba:", numba.__version__)
print("SHAP:", shap.__version__)

print("\nSHAP imported successfully.")

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import shap
import numba
import xgboost as xgb

from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    120
)

print(
    "Scikit-learn:",
    sklearn.__version__
)

print(
    "SHAP:",
    shap.__version__
)

print(
    "XGBoost:",
    xgb.__version__
)

print(
    "NumPy:",
    np.__version__
)

print(
    "Numba:",
    numba.__version__
)

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """
    Find the project directory containing the
    processed customer-churn dataset.
    """

    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:

        expected_dataset = (
            candidate
            / "data"
            / "processed"
            / "telco_churn_model.csv"
        )

        if expected_dataset.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find "
        "data/processed/telco_churn_model.csv. "
        "Open this notebook from inside the "
        "customer-churn-project directory."
    )


project_root = find_project_root(Path.cwd())

models_directory = project_root / "models"

model_output_directory = (
    project_root
    / "reports"
    / "model_results"
)

explainability_output_directory = (
    project_root
    / "reports"
    / "explainability"
)

figures_directory = (
    project_root
    / "reports"
    / "figures"
    / "explainability"
)

explainability_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

figures_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)

In [ ]:
model_data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_model.csv"
)

model_df = pd.read_csv(model_data_path)

target_column = "churn_value"

assert target_column in model_df.columns

X = model_df.drop(
    columns=[target_column]
).copy()

y = model_df[
    target_column
].astype(int).copy()

print("Model data shape:", model_df.shape)
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Churn prevalence:", round(y.mean(), 4))

display(model_df.head())

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print(
    "Training churn rate:",
    round(y_train.mean(), 4)
)

print(
    "Test churn rate:",
    round(y_test.mean(), 4)
)

In [ ]:
final_model_path = (
    models_directory
    / "final_tuned_churn_model.joblib"
)

base_model_path = (
    models_directory
    / "best_tuned_cv_model.joblib"
)

metadata_path = (
    models_directory
    / "final_tuned_churn_model_metadata.json"
)

for required_path in [
    final_model_path,
    base_model_path,
    metadata_path
]:
    if not required_path.exists():

        raise FileNotFoundError(
            f"Required artifact is missing: "
            f"{required_path}"
        )


final_model = joblib.load(
    final_model_path
)

best_tuned_base_model = joblib.load(
    base_model_path
)

with open(
    metadata_path,
    "r",
    encoding="utf-8"
) as file:

    final_model_metadata = json.load(file)


selected_model_name = (
    final_model_metadata["model_name"]
)

selected_threshold = float(
    final_model_metadata["selected_threshold"]
)

probability_source = (
    final_model_metadata["probability_source"]
)


print("Selected model:", selected_model_name)

print(
    "Probability source:",
    probability_source
)

print(
    "Selected threshold:",
    selected_threshold
)

print("\nBase-model pipeline:")
print(
    best_tuned_base_model.named_steps.keys()
)

In [ ]:
test_probabilities = (
    final_model.predict_proba(
        X_test
    )[:, 1]
)

test_predictions = (
    test_probabilities
    >=
    selected_threshold
).astype(int)


print(
    "Number of test probabilities:",
    len(test_probabilities)
)

print(
    "Minimum probability:",
    round(test_probabilities.min(), 4)
)

print(
    "Maximum probability:",
    round(test_probabilities.max(), 4)
)

print(
    "Predicted churn rate:",
    round(test_predictions.mean(), 4)
)

print(
    "Actual churn rate:",
    round(y_test.mean(), 4)
)

In [ ]:
permutation_results = permutation_importance(
    estimator=final_model,

    X=X_test,
    y=y_test,

    # Same primary metric used during model selection
    scoring="average_precision",

    n_repeats=20,

    random_state=42,

    # Safer for GPU models and calibrated wrappers
    n_jobs=1
)


permutation_importance_df = pd.DataFrame(
    {
        "feature": X_test.columns,

        "importance_mean": (
            permutation_results.importances_mean
        ),

        "importance_std": (
            permutation_results.importances_std
        )
    }
)


permutation_importance_df = (
    permutation_importance_df
    .sort_values(
        "importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    permutation_importance_df.round(5)
)

In [ ]:
top_permutation_features = (
    permutation_importance_df
    .head(15)
    .sort_values(
        "importance_mean",
        ascending=True
    )
)


plt.figure(figsize=(9, 7))

plt.barh(
    top_permutation_features["feature"],
    top_permutation_features[
        "importance_mean"
    ],

    xerr=top_permutation_features[
        "importance_std"
    ]
)

plt.xlabel(
    "Decrease in Test PR-AUC After Shuffling"
)

plt.ylabel("Feature")

plt.title(
    "Permutation Importance — Final Churn Model"
)

plt.tight_layout()


permutation_figure_path = (
    figures_directory
    / "permutation_importance.png"
)

plt.savefig(
    permutation_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
base_preprocessor = (
    best_tuned_base_model
    .named_steps["preprocessor"]
)

base_classifier = (
    best_tuned_base_model
    .named_steps["classifier"]
)


encoded_feature_names = (
    base_preprocessor
    .get_feature_names_out()
)


print(
    "Base classifier type:",
    type(base_classifier).__name__
)

print(
    "Number of original features:",
    X_train.shape[1]
)

print(
    "Number of encoded features:",
    len(encoded_feature_names)
)

In [ ]:
def clean_encoded_feature_name(
    feature_name
):
    """
    Remove ColumnTransformer prefixes.
    """

    if "__" in feature_name:

        return feature_name.split(
            "__",
            maxsplit=1
        )[1]

    return feature_name


readable_encoded_feature_names = [
    clean_encoded_feature_name(name)

    for name in encoded_feature_names
]


print(
    readable_encoded_feature_names[:20]
)

In [ ]:
if isinstance(
    base_classifier,
    LogisticRegression
):

    model_specific_importance = pd.DataFrame(
        {
            "encoded_feature": (
                readable_encoded_feature_names
            ),

            "importance_value": (
                base_classifier.coef_[0]
            )
        }
    )

    model_specific_importance[
        "absolute_importance"
    ] = (
        model_specific_importance[
            "importance_value"
        ].abs()
    )

    model_specific_importance[
        "effect_direction"
    ] = np.where(
        model_specific_importance[
            "importance_value"
        ] > 0,

        "Raises predicted churn risk",

        "Lowers predicted churn risk"
    )

    importance_method = (
        "Logistic Regression coefficient"
    )


elif isinstance(
    base_classifier,
    (
        RandomForestClassifier,
        XGBClassifier
    )
):

    model_specific_importance = pd.DataFrame(
        {
            "encoded_feature": (
                readable_encoded_feature_names
            ),

            "importance_value": (
                base_classifier
                .feature_importances_
            )
        }
    )

    model_specific_importance[
        "absolute_importance"
    ] = (
        model_specific_importance[
            "importance_value"
        ]
    )

    model_specific_importance[
        "effect_direction"
    ] = (
        "Direction not provided by "
        "built-in tree importance"
    )

    importance_method = (
        "Built-in tree feature importance"
    )


else:

    model_specific_importance = pd.DataFrame()

    importance_method = (
        "No supported model-specific method"
    )


if not model_specific_importance.empty:

    model_specific_importance = (
        model_specific_importance
        .sort_values(
            "absolute_importance",
            ascending=False
        )
        .reset_index(drop=True)
    )

    display(
        model_specific_importance
        .head(25)
        .round(5)
    )

else:

    print(
        "Model-specific importance "
        "was not generated."
    )


print("\nImportance method:")
print(importance_method)

In [ ]:
if isinstance(
    base_classifier,
    LogisticRegression
):

    positive_coefficients = (
        model_specific_importance[
            model_specific_importance[
                "importance_value"
            ] > 0
        ]
        .sort_values(
            "importance_value",
            ascending=False
        )
        .head(15)
    )


    negative_coefficients = (
        model_specific_importance[
            model_specific_importance[
                "importance_value"
            ] < 0
        ]
        .sort_values(
            "importance_value",
            ascending=True
        )
        .head(15)
    )


    print(
        "Features associated with "
        "higher predicted churn:"
    )

    display(
        positive_coefficients[
            [
                "encoded_feature",
                "importance_value"
            ]
        ].round(4)
    )


    print(
        "\nFeatures associated with "
        "lower predicted churn:"
    )

    display(
        negative_coefficients[
            [
                "encoded_feature",
                "importance_value"
            ]
        ].round(4)
    )

else:

    print(
        "Coefficient analysis skipped "
        "because the selected model "
        "is not Logistic Regression."
    )

In [ ]:
if not model_specific_importance.empty:

    top_model_specific_features = (
        model_specific_importance
        .head(20)
        .sort_values(
            "absolute_importance",
            ascending=True
        )
    )


    plt.figure(figsize=(10, 8))

    plt.barh(
        top_model_specific_features[
            "encoded_feature"
        ],

        top_model_specific_features[
            "importance_value"
        ]
    )

    plt.xlabel(
        importance_method
    )

    plt.ylabel(
        "Encoded Feature"
    )

    plt.title(
        f"Model-Specific Importance\n"
        f"{selected_model_name}"
    )

    plt.tight_layout()


    model_specific_figure_path = (
        figures_directory
        / "model_specific_importance.png"
    )

    plt.savefig(
        model_specific_figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
numeric_columns = (
    X_train
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)


categorical_columns = [
    column

    for column in X_train.columns

    if column not in numeric_columns
]


original_feature_order = (
    X_train.columns.tolist()
)


def restore_feature_dataframe(data):
    """
    Convert SHAP's masked input back into a
    DataFrame with the correct column names
    and data types.
    """

    if isinstance(data, pd.DataFrame):

        restored_df = data.copy()

    else:

        restored_df = pd.DataFrame(
            data,
            columns=original_feature_order
        )


    restored_df = restored_df[
        original_feature_order
    ].copy()


    for column in numeric_columns:

        restored_df[column] = pd.to_numeric(
            restored_df[column],
            errors="coerce"
        )


    for column in categorical_columns:

        restored_df[column] = (
            restored_df[column]
            .astype(object)
        )


    return restored_df


def deployed_churn_probability(data):
    """
    Return only P(churn = 1) from the complete
    final evaluation model.
    """

    restored_df = restore_feature_dataframe(
        data
    )

    return final_model.predict_proba(
        restored_df
    )[:, 1]

In [ ]:
prediction_function_test = (
    deployed_churn_probability(
        X_test.iloc[:5]
    )
)


direct_prediction_test = (
    final_model.predict_proba(
        X_test.iloc[:5]
    )[:, 1]
)


print(
    "Wrapper probabilities:"
)

print(
    prediction_function_test
)


print(
    "\nDirect probabilities:"
)

print(
    direct_prediction_test
)


assert np.allclose(
    prediction_function_test,
    direct_prediction_test
)


print(
    "\nPrediction wrapper test passed."
)

In [ ]:
SHAP_BACKGROUND_SIZE = 100

SHAP_EXPLANATION_SIZE = min(
    300,
    len(X_test)
)

NUMBER_OF_TOP_RISK_CUSTOMERS = min(
    25,
    SHAP_EXPLANATION_SIZE
)


background_X = (
    X_train
    .sample(
        n=min(
            SHAP_BACKGROUND_SIZE,
            len(X_train)
        ),
        random_state=42
    )
    .copy()
)


sorted_test_positions = np.argsort(
    -test_probabilities
)


top_risk_positions = (
    sorted_test_positions[
        :NUMBER_OF_TOP_RISK_CUSTOMERS
    ]
)


all_test_positions = np.arange(
    len(X_test)
)


remaining_positions = np.setdiff1d(
    all_test_positions,
    top_risk_positions
)


remaining_sample_size = (
    SHAP_EXPLANATION_SIZE
    -
    len(top_risk_positions)
)


random_generator = np.random.default_rng(
    42
)


if remaining_sample_size > 0:

    random_positions = (
        random_generator.choice(
            remaining_positions,
            size=remaining_sample_size,
            replace=False
        )
    )

else:

    random_positions = np.array(
        [],
        dtype=int
    )


explanation_positions = np.concatenate(
    [
        top_risk_positions,
        random_positions
    ]
)


X_explain = (
    X_test
    .iloc[explanation_positions]
    .copy()
)


explanation_probabilities = (
    test_probabilities[
        explanation_positions
    ]
)


print(
    "Background rows:",
    background_X.shape[0]
)

print(
    "Customers explained:",
    X_explain.shape[0]
)

print(
    "Features explained:",
    X_explain.shape[1]
)

In [ ]:
shap_explainer = None

try:
    shap_explainer = shap.explainers.Permutation(
        model=deployed_churn_probability,
        masker=background_X,
        feature_names=original_feature_order,
        seed=42
    )
except TypeError:
    shap_explainer = shap.explainers.Permutation(
        model=deployed_churn_probability,
        masker=background_X,
        seed=42
    )

minimum_shap_evaluations = (
    2 * X_explain.shape[1] + 1
)

print(
    "Minimum evaluations per customer:",
    minimum_shap_evaluations
)


In [ ]:
print(
    "Final prediction model:",
    selected_model_name
)

print(
    "Final probability source:",
    probability_source
)

print(
    "\nPermutation importance already explains "
    "the complete final calibrated pipeline."
)

print(
    "SHAP will explain the underlying tuned "
    "XGBoost probability model before calibration."
)

In [ ]:
if not isinstance(
    base_classifier,
    XGBClassifier
):

    raise TypeError(
        "This corrected SHAP section expects "
        "XGBoost because XGBoost was selected "
        "in your Notebook 4 results. "
        f"Current classifier: "
        f"{type(base_classifier).__name__}"
    )


print(
    "XGBoost classifier confirmed."
)

print(
    "Encoded feature count:",
    len(encoded_feature_names)
)

In [ ]:
from scipy import sparse


def make_dense_numeric(matrix):
    """
    Convert the preprocessor output into a
    dense numeric NumPy array for SHAP.
    """

    if sparse.issparse(matrix):

        matrix = matrix.toarray()

    return np.asarray(
        matrix,
        dtype=np.float64
    )

In [ ]:
SHAP_BACKGROUND_SIZE = 100

SHAP_EXPLANATION_SIZE = min(
    300,
    len(X_test)
)

NUMBER_OF_TOP_RISK_CUSTOMERS = min(
    25,
    SHAP_EXPLANATION_SIZE
)


background_X_raw = (
    X_train
    .sample(
        n=min(
            SHAP_BACKGROUND_SIZE,
            len(X_train)
        ),
        random_state=42
    )
    .copy()
)


sorted_test_positions = np.argsort(
    -test_probabilities
)


top_risk_positions = (
    sorted_test_positions[
        :NUMBER_OF_TOP_RISK_CUSTOMERS
    ]
)


all_test_positions = np.arange(
    len(X_test)
)


remaining_positions = np.setdiff1d(
    all_test_positions,
    top_risk_positions
)


remaining_sample_size = (
    SHAP_EXPLANATION_SIZE
    -
    len(top_risk_positions)
)


random_generator = np.random.default_rng(
    42
)


if remaining_sample_size > 0:

    random_positions = (
        random_generator.choice(
            remaining_positions,
            size=remaining_sample_size,
            replace=False
        )
    )

else:

    random_positions = np.array(
        [],
        dtype=int
    )


explanation_positions = np.concatenate(
    [
        top_risk_positions,
        random_positions
    ]
)


X_explain_raw = (
    X_test
    .iloc[explanation_positions]
    .copy()
)


final_calibrated_probabilities_explained = (
    test_probabilities[
        explanation_positions
    ]
)


print(
    "Background customers:",
    len(background_X_raw)
)

print(
    "Customers selected for SHAP:",
    len(X_explain_raw)
)

print(
    "Original feature count:",
    X_explain_raw.shape[1]
)

In [ ]:
X_background_encoded = (
    base_preprocessor.transform(
        background_X_raw
    )
)


X_explain_encoded = (
    base_preprocessor.transform(
        X_explain_raw
    )
)


X_background_encoded = make_dense_numeric(
    X_background_encoded
)


X_explain_encoded = make_dense_numeric(
    X_explain_encoded
)


print(
    "Encoded background shape:",
    X_background_encoded.shape
)

print(
    "Encoded explanation shape:",
    X_explain_encoded.shape
)

print(
    "Encoded feature names:",
    len(readable_encoded_feature_names)
)


assert (
    X_background_encoded.shape[1]
    ==
    len(readable_encoded_feature_names)
)


assert (
    X_explain_encoded.shape[1]
    ==
    len(readable_encoded_feature_names)
)

In [ ]:
base_xgboost_probabilities = (
    base_classifier.predict_proba(
        X_explain_encoded
    )[:, 1]
)


pipeline_base_probabilities = (
    best_tuned_base_model.predict_proba(
        X_explain_raw
    )[:, 1]
)


assert np.allclose(
    base_xgboost_probabilities,
    pipeline_base_probabilities,
    atol=1e-7
)


probability_comparison = pd.DataFrame(
    {
        "base_xgboost_probability": (
            base_xgboost_probabilities[:10]
        ),

        "final_calibrated_probability": (
            final_calibrated_probabilities_explained[
                :10
            ]
        )
    }
)


display(
    probability_comparison.round(5)
)


print(
    "Base-model prediction check passed."
)

In [ ]:
# SHAP 0.52 cannot use TreeExplainer directly with an XGBoost model
# whose saved configuration has enable_categorical=True. XGBoost's native
# pred_contribs implementation computes the same exact TreeSHAP values and
# supports this model configuration.
xgboost_booster = base_classifier.get_booster()

print("Using XGBoost native TreeSHAP contributions.")


In [ ]:
explanation_matrix = xgb.DMatrix(
    X_explain_encoded
)

native_tree_shap_contributions = xgboost_booster.predict(
    explanation_matrix,
    pred_contribs=True,
    validate_features=False
)

expected_shape = (
    len(X_explain_encoded),
    X_explain_encoded.shape[1] + 1
)

if native_tree_shap_contributions.shape != expected_shape:
    raise ValueError(
        "Unexpected native TreeSHAP shape "
        f"{native_tree_shap_contributions.shape}; "
        f"expected {expected_shape}."
    )

# The final column is XGBoost's bias (base-value) contribution.
encoded_shap_array = native_tree_shap_contributions[:, :-1]
shap_base_values = native_tree_shap_contributions[:, -1]

explained_raw_margins = (
    shap_base_values + encoded_shap_array.sum(axis=1)
)

model_raw_margins = xgboost_booster.predict(
    explanation_matrix,
    output_margin=True,
    validate_features=False
)

assert np.allclose(
    explained_raw_margins,
    model_raw_margins,
    atol=1e-5
)

print("SHAP array shape:", encoded_shap_array.shape)
print("Base values shape:", shap_base_values.shape)
print("SHAP output scale: raw XGBoost margin (log-odds)")
print("SHAP additivity check passed.")


In [ ]:
from scipy.special import expit


shap_reconstructed_base_probabilities = (
    expit(explained_raw_margins)
)


maximum_probability_reconstruction_error = float(
    np.max(
        np.abs(
            shap_reconstructed_base_probabilities
            -
            base_xgboost_probabilities
        )
    )
)


probability_reconstruction_check = pd.DataFrame(
    {
        "xgboost_probability": (
            base_xgboost_probabilities[:10]
        ),

        "shap_reconstructed_probability": (
            shap_reconstructed_base_probabilities[:10]
        ),

        "absolute_error": np.abs(
            base_xgboost_probabilities[:10]
            -
            shap_reconstructed_base_probabilities[:10]
        ),

        "final_calibrated_probability": (
            final_calibrated_probabilities_explained[:10]
        )
    }
)


display(
    probability_reconstruction_check.round(7)
)


print(
    "Maximum probability reconstruction error:",
    maximum_probability_reconstruction_error
)


assert np.allclose(
    shap_reconstructed_base_probabilities,
    base_xgboost_probabilities,
    atol=1e-5
)


print(
    "SHAP probability reconstruction passed."
)

In [ ]:
encoded_shap_explanation = shap.Explanation(
    values=encoded_shap_array,

    base_values=shap_base_values,

    data=X_explain_encoded,

    feature_names=(
        readable_encoded_feature_names
    )
)


print(
    "SHAP Explanation shape:",
    encoded_shap_explanation.values.shape
)

print(
    "SHAP output scale:"
)

print(
    "Raw XGBoost margin / log-odds"
)

In [ ]:
encoded_shap_global_importance = pd.DataFrame(
    {
        "encoded_feature": (
            readable_encoded_feature_names
        ),

        "mean_absolute_shap": (
            np.abs(
                encoded_shap_array
            ).mean(axis=0)
        ),

        "mean_signed_shap": (
            encoded_shap_array.mean(axis=0)
        )
    }
)


encoded_shap_global_importance = (
    encoded_shap_global_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    encoded_shap_global_importance
    .head(25)
    .round(6)
)

In [ ]:
encoded_shap_global_importance = pd.DataFrame(
    {
        "encoded_feature": (
            readable_encoded_feature_names
        ),

        "mean_absolute_shap": (
            np.abs(
                encoded_shap_array
            ).mean(axis=0)
        ),

        "mean_signed_shap": (
            encoded_shap_array.mean(axis=0)
        )
    }
)


encoded_shap_global_importance = (
    encoded_shap_global_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    encoded_shap_global_importance
    .head(25)
    .round(6)
)

In [ ]:
shap.plots.beeswarm(
    encoded_shap_explanation,

    max_display=20,

    show=False
)


plt.title(
    "SHAP Impact on XGBoost Churn Margin"
)

plt.tight_layout()


shap_beeswarm_figure_path = (
    figures_directory
    / "shap_encoded_beeswarm.png"
)


plt.savefig(
    shap_beeswarm_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
original_feature_order = (
    X_train.columns.tolist()
)


def map_encoded_to_original(
    encoded_feature_name
):
    """
    Map each encoded feature back to its
    original customer feature.
    """

    matching_features = [
        original_feature

        for original_feature
        in original_feature_order

        if (
            encoded_feature_name
            ==
            original_feature
        )
        or
        encoded_feature_name.startswith(
            original_feature + "_"
        )
    ]


    if not matching_features:

        return encoded_feature_name


    # Longest match prevents ambiguous matches
    return max(
        matching_features,
        key=len
    )


encoded_to_original_mapping = [
    map_encoded_to_original(
        encoded_feature_name
    )

    for encoded_feature_name
    in readable_encoded_feature_names
]


feature_mapping_df = pd.DataFrame(
    {
        "encoded_feature": (
            readable_encoded_feature_names
        ),

        "original_feature": (
            encoded_to_original_mapping
        )
    }
)


display(
    feature_mapping_df
)

In [ ]:
raw_shap_matrix = np.zeros(
    (
        len(X_explain_raw),
        len(original_feature_order)
    ),
    dtype=float
)


for raw_feature_position, raw_feature in enumerate(
    original_feature_order
):

    encoded_positions = [
        position

        for position, mapped_feature
        in enumerate(
            encoded_to_original_mapping
        )

        if mapped_feature == raw_feature
    ]


    if encoded_positions:

        raw_shap_matrix[
            :,
            raw_feature_position
        ] = encoded_shap_array[
            :,
            encoded_positions
        ].sum(axis=1)


print(
    "Original-feature SHAP matrix shape:",
    raw_shap_matrix.shape
)


assert raw_shap_matrix.shape == (
    len(X_explain_raw),
    len(original_feature_order)
)


# Aggregating features must preserve total SHAP contribution
assert np.allclose(
    raw_shap_matrix.sum(axis=1),
    encoded_shap_array.sum(axis=1),
    atol=1e-7
)


print(
    "Original-feature aggregation check passed."
)

In [ ]:
raw_shap_global_importance = pd.DataFrame(
    {
        "feature": (
            original_feature_order
        ),

        "mean_absolute_shap": (
            np.abs(
                raw_shap_matrix
            ).mean(axis=0)
        ),

        "mean_signed_shap": (
            raw_shap_matrix.mean(axis=0)
        )
    }
)


raw_shap_global_importance = (
    raw_shap_global_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    raw_shap_global_importance.round(6)
)

In [ ]:
top_raw_shap_features = (
    raw_shap_global_importance
    .head(15)
    .sort_values(
        "mean_absolute_shap",
        ascending=True
    )
)


plt.figure(figsize=(9, 7))


plt.barh(
    top_raw_shap_features["feature"],

    top_raw_shap_features[
        "mean_absolute_shap"
    ]
)


plt.xlabel(
    "Mean Absolute SHAP Contribution"
)

plt.ylabel(
    "Original Customer Feature"
)

plt.title(
    "Global SHAP Importance — Original Features"
)

plt.tight_layout()


raw_shap_figure_path = (
    figures_directory
    / "shap_original_feature_importance.png"
)


plt.savefig(
    raw_shap_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
enriched_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics_enriched.csv"
)


base_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics.csv"
)


if enriched_analytics_path.exists():

    analytics_df = pd.read_csv(
        enriched_analytics_path
    )

elif base_analytics_path.exists():

    analytics_df = pd.read_csv(
        base_analytics_path
    )

else:

    raise FileNotFoundError(
        "No analytics dataset was found."
    )


assert len(analytics_df) == len(model_df)

assert "customerid" in analytics_df.columns


print(
    "Analytics data shape:",
    analytics_df.shape
)

In [ ]:
highest_risk_position = 0


highest_risk_original_index = (
    X_explain_raw.index[
        highest_risk_position
    ]
)


highest_risk_customer_id = (
    analytics_df.loc[
        highest_risk_original_index,
        "customerid"
    ]
)


highest_risk_base_probability = float(
    base_xgboost_probabilities[
        highest_risk_position
    ]
)


highest_risk_final_probability = float(
    final_calibrated_probabilities_explained[
        highest_risk_position
    ]
)


print(
    "Customer ID:",
    highest_risk_customer_id
)

print(
    "Actual churn:",
    int(
        y_test.loc[
            highest_risk_original_index
        ]
    )
)

print(
    "Base XGBoost probability:",
    round(
        highest_risk_base_probability,
        4
    )
)

print(
    "Final calibrated probability:",
    round(
        highest_risk_final_probability,
        4
    )
)

print(
    "Selected threshold:",
    selected_threshold
)


shap.plots.waterfall(
    encoded_shap_explanation[
        highest_risk_position
    ],

    max_display=15,

    show=False
)


plt.title(
    f"XGBoost SHAP Explanation\n"
    f"Customer: {highest_risk_customer_id}"
)

plt.tight_layout()


waterfall_figure_path = (
    figures_directory
    / "highest_risk_customer_waterfall.png"
)


plt.savefig(
    waterfall_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
highest_risk_raw_shap = pd.Series(
    raw_shap_matrix[
        highest_risk_position
    ],

    index=original_feature_order
)


highest_risk_local_drivers = pd.DataFrame(
    {
        "feature": (
            original_feature_order
        ),

        "feature_value": [
            X_explain_raw.loc[
                highest_risk_original_index,
                feature
            ]

            for feature
            in original_feature_order
        ],

        "shap_log_odds_contribution": (
            highest_risk_raw_shap.values
        )
    }
)


highest_risk_local_drivers[
    "absolute_contribution"
] = (
    highest_risk_local_drivers[
        "shap_log_odds_contribution"
    ].abs()
)


highest_risk_local_drivers[
    "effect"
] = np.where(
    highest_risk_local_drivers[
        "shap_log_odds_contribution"
    ] > 0,

    "Raises churn risk",

    "Lowers churn risk"
)


highest_risk_local_drivers = (
    highest_risk_local_drivers
    .sort_values(
        "absolute_contribution",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    highest_risk_local_drivers
    .head(12)
    .round(6)
)

In [ ]:
top_highest_risk_drivers = (
    highest_risk_local_drivers
    .head(12)
    .sort_values(
        "shap_log_odds_contribution",
        ascending=True
    )
)


plt.figure(figsize=(9, 7))


plt.barh(
    top_highest_risk_drivers["feature"],

    top_highest_risk_drivers[
        "shap_log_odds_contribution"
    ]
)


plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel(
    "SHAP Contribution to XGBoost Log-Odds"
)

plt.ylabel("Original Feature")

plt.title(
    f"Top Churn Drivers — {highest_risk_customer_id}"
)

plt.tight_layout()


local_driver_figure_path = (
    figures_directory
    / "highest_risk_original_feature_drivers.png"
)


plt.savefig(
    local_driver_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
reason_code_rows = []


for customer_position, customer_index in enumerate(
    X_explain_raw.index
):

    customer_raw_shap = pd.Series(
        raw_shap_matrix[
            customer_position
        ],

        index=original_feature_order
    )


    positive_drivers = (
        customer_raw_shap[
            customer_raw_shap > 0
        ]
        .sort_values(
            ascending=False
        )
        .head(3)
    )


    negative_drivers = (
        customer_raw_shap[
            customer_raw_shap < 0
        ]
        .sort_values(
            ascending=True
        )
        .head(2)
    )


    customer_id = analytics_df.loc[
        customer_index,
        "customerid"
    ]


    final_probability = float(
        final_calibrated_probabilities_explained[
            customer_position
        ]
    )


    base_probability = float(
        base_xgboost_probabilities[
            customer_position
        ]
    )


    common_customer_data = {
        "customer_index": int(
            customer_index
        ),

        "customerid": customer_id,

        "base_xgboost_probability": (
            base_probability
        ),

        "final_calibrated_probability": (
            final_probability
        ),

        "actual_churn": int(
            y_test.loc[
                customer_index
            ]
        ),

        "predicted_churn": int(
            final_probability
            >=
            selected_threshold
        )
    }


    for rank, (
        feature_name,
        shap_contribution
    ) in enumerate(
        positive_drivers.items(),
        start=1
    ):

        reason_code_rows.append(
            {
                **common_customer_data,

                "driver_type": (
                    "Raises churn risk"
                ),

                "driver_rank": rank,

                "feature": feature_name,

                "feature_value": (
                    X_explain_raw.loc[
                        customer_index,
                        feature_name
                    ]
                ),

                "shap_log_odds_contribution": (
                    float(
                        shap_contribution
                    )
                )
            }
        )


    for rank, (
        feature_name,
        shap_contribution
    ) in enumerate(
        negative_drivers.items(),
        start=1
    ):

        reason_code_rows.append(
            {
                **common_customer_data,

                "driver_type": (
                    "Lowers churn risk"
                ),

                "driver_rank": rank,

                "feature": feature_name,

                "feature_value": (
                    X_explain_raw.loc[
                        customer_index,
                        feature_name
                    ]
                ),

                "shap_log_odds_contribution": (
                    float(
                        shap_contribution
                    )
                )
            }
        )


customer_reason_codes = pd.DataFrame(
    reason_code_rows
)


display(
    customer_reason_codes.head(15)
)

In [ ]:
explained_customer_base = pd.DataFrame(
    {
        "customer_index": (
            X_explain_raw.index.astype(int)
        ),

        "customerid": (
            analytics_df.loc[
                X_explain_raw.index,
                "customerid"
            ].values
        ),

        "actual_churn": (
            y_test.loc[
                X_explain_raw.index
            ].values
        ),

        "base_xgboost_probability": (
            base_xgboost_probabilities
        ),

        "final_calibrated_probability": (
            final_calibrated_probabilities_explained
        ),

        "predicted_churn": (
            final_calibrated_probabilities_explained
            >=
            selected_threshold
        ).astype(int)
    }
)

In [ ]:
positive_reason_codes = (
    customer_reason_codes[
        customer_reason_codes[
            "driver_type"
        ]
        ==
        "Raises churn risk"
    ]
    .copy()
)


positive_reason_codes[
    "reason_text"
] = (
    positive_reason_codes["feature"]
    +
    "="
    +
    positive_reason_codes[
        "feature_value"
    ].astype(str)
)


risk_reason_text = (
    positive_reason_codes
    .sort_values(
        [
            "customerid",
            "driver_rank"
        ]
    )
    .groupby(
        "customerid"
    )[
        "reason_text"
    ]
    .apply(
        lambda values: " | ".join(values)
    )
    .reset_index(
        name="top_risk_reasons"
    )
)


customer_reason_summary = (
    explained_customer_base
    .merge(
        risk_reason_text,
        on="customerid",
        how="left"
    )
)


customer_reason_summary[
    "top_risk_reasons"
] = (
    customer_reason_summary[
        "top_risk_reasons"
    ]
    .fillna(
        "No positive SHAP driver identified"
    )
)


display(
    customer_reason_summary
    .sort_values(
        "final_calibrated_probability",
        ascending=False
    )
    .head(20)
)

In [ ]:
fairness_audit_df = X_test.copy()


fairness_audit_df[
    "actual_churn"
] = y_test.loc[
    X_test.index
].values


fairness_audit_df[
    "churn_probability"
] = test_probabilities


fairness_audit_df[
    "predicted_churn"
] = test_predictions


protected_features = [
    feature

    for feature in [
        "gender",
        "senior_citizen"
    ]

    if feature in fairness_audit_df.columns
]


print(
    "Protected features available:",
    protected_features
)

In [ ]:
def safe_divide(
    numerator,
    denominator
):

    if denominator == 0:

        return np.nan

    return numerator / denominator


def calculate_group_metrics(
    group_df
):

    y_true = group_df[
        "actual_churn"
    ].astype(int)

    y_pred = group_df[
        "predicted_churn"
    ].astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()


    return {
        "customer_count": len(group_df),

        "actual_churn_rate": (
            y_true.mean()
        ),

        "mean_predicted_probability": (
            group_df[
                "churn_probability"
            ].mean()
        ),

        "predicted_positive_rate": (
            y_pred.mean()
        ),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "false_positive_rate": safe_divide(
            fp,
            fp + tn
        ),

        "false_negative_rate": safe_divide(
            fn,
            fn + tp
        ),

        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }

In [ ]:
fairness_rows = []


for protected_feature in protected_features:

    grouped_data = fairness_audit_df.groupby(
        protected_feature,
        dropna=False
    )


    for group_value, group_df in grouped_data:

        group_metrics = (
            calculate_group_metrics(
                group_df
            )
        )


        fairness_rows.append(
            {
                "protected_feature": (
                    protected_feature
                ),

                "group": str(
                    group_value
                ),

                **group_metrics
            }
        )


fairness_group_metrics = pd.DataFrame(
    fairness_rows
)


display(
    fairness_group_metrics.round(4)
)

In [ ]:
for protected_feature in protected_features:

    feature_metrics = (
        fairness_group_metrics[
            fairness_group_metrics[
                "protected_feature"
            ]
            ==
            protected_feature
        ]
        .copy()
    )


    fairness_plot_data = (
        feature_metrics
        .set_index("group")[
            [
                "recall",
                "false_positive_rate"
            ]
        ]
    )


    fairness_plot_data.plot(
        kind="bar",
        figsize=(8, 5)
    )


    plt.title(
        f"Group Performance — "
        f"{protected_feature}"
    )

    plt.xlabel("Group")
    plt.ylabel("Metric")

    plt.ylim(0, 1)
    plt.xticks(rotation=0)

    plt.tight_layout()


    fairness_figure_path = (
        figures_directory
        /
        f"fairness_{protected_feature}.png"
    )


    plt.savefig(
        fairness_figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
explanation_method_comparison = (
    permutation_importance_df[
        [
            "feature",
            "importance_mean"
        ]
    ]
    .merge(
        raw_shap_global_importance[
            [
                "feature",
                "mean_absolute_shap"
            ]
        ],

        on="feature",

        how="outer"
    )
)


explanation_method_comparison[
    "permutation_rank"
] = (
    explanation_method_comparison[
        "importance_mean"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


explanation_method_comparison[
    "shap_rank"
] = (
    explanation_method_comparison[
        "mean_absolute_shap"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


explanation_method_comparison[
    "rank_difference"
] = (
    explanation_method_comparison[
        "permutation_rank"
    ]
    -
    explanation_method_comparison[
        "shap_rank"
    ]
).abs()


explanation_method_comparison = (
    explanation_method_comparison
    .sort_values(
        "shap_rank"
    )
    .reset_index(drop=True)
)


display(
    explanation_method_comparison.round(5)
)

In [ ]:
permutation_importance_df.to_csv(
    explainability_output_directory
    / "permutation_importance.csv",

    index=False
)


encoded_shap_global_importance.to_csv(
    explainability_output_directory
    / "shap_encoded_global_importance.csv",

    index=False
)


raw_shap_global_importance.to_csv(
    explainability_output_directory
    / "shap_original_feature_importance.csv",

    index=False
)


feature_mapping_df.to_csv(
    explainability_output_directory
    / "encoded_to_original_feature_mapping.csv",

    index=False
)


highest_risk_local_drivers.to_csv(
    explainability_output_directory
    / "highest_risk_customer_drivers.csv",

    index=False
)


customer_reason_codes.to_csv(
    explainability_output_directory
    / "customer_reason_codes.csv",

    index=False
)


customer_reason_summary.to_csv(
    explainability_output_directory
    / "customer_reason_summary.csv",

    index=False
)


fairness_group_metrics.to_csv(
    explainability_output_directory
    / "fairness_group_metrics.csv",

    index=False
)


explanation_method_comparison.to_csv(
    explainability_output_directory
    / "explanation_method_comparison.csv",

    index=False
)


if not model_specific_importance.empty:

    model_specific_importance.to_csv(
        explainability_output_directory
        / "model_specific_importance.csv",

        index=False
    )


print(
    "Explainability tables saved to:"
)

print(
    explainability_output_directory
)

In [ ]:
explained_customer_predictions = (
    customer_reason_summary.copy()
)


explained_customer_predictions[
    "selected_threshold"
] = selected_threshold


explained_customer_predictions[
    "shap_base_margin"
] = shap_base_values


explained_customer_predictions[
    "shap_reconstructed_margin"
] = explained_raw_margins


explained_customer_predictions[
    "shap_reconstructed_base_probability"
] = (
    shap_reconstructed_base_probabilities
)


explained_customer_predictions.to_csv(
    explainability_output_directory
    / "explained_customer_predictions.csv",

    index=False
)


display(
    explained_customer_predictions
    .sort_values(
        "final_calibrated_probability",
        ascending=False
    )
    .head(20)
)

In [ ]:
explainability_metadata = {
    "notebook": (
        "06_model_explainability"
    ),

    "final_model_name": (
        selected_model_name
    ),

    "final_probability_source": (
        probability_source
    ),

    "selected_threshold": float(
        selected_threshold
    ),

    "permutation_importance_scope": (
        "Complete final calibrated pipeline"
    ),

    "permutation_importance_metric": (
        "Average precision / PR-AUC"
    ),

    "tree_shap_scope": (
        "Underlying tuned XGBoost classifier"
    ),

    "tree_shap_implementation": (
        "XGBoost native pred_contribs"
    ),

    "tree_shap_output_scale": (
        "Raw XGBoost margin / log-odds"
    ),

    "number_of_original_features": int(
        len(original_feature_order)
    ),

    "number_of_encoded_features": int(
        len(readable_encoded_feature_names)
    ),

    "number_of_explained_customers": int(
        len(X_explain_raw)
    ),

    "maximum_probability_reconstruction_error": (
        maximum_probability_reconstruction_error
    ),

    "top_permutation_feature": (
        permutation_importance_df.loc[
            0,
            "feature"
        ]
    ),

    "top_shap_feature": (
        raw_shap_global_importance.loc[
            0,
            "feature"
        ]
    ),

    "protected_features_audited": (
        protected_features
    ),

    "causal_interpretation": False,

    "scikit_learn_version": (
        sklearn.__version__
    ),

    "shap_version": (
        shap.__version__
    ),

    "xgboost_version": (
        xgb.__version__
    )
}


explainability_metadata_path = (
    explainability_output_directory
    / "explainability_metadata.json"
)


with open(
    explainability_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        explainability_metadata,
        file,
        indent=4
    )


print(
    "Metadata saved to:"
)

print(
    explainability_metadata_path
)

In [ ]:
required_explainability_artifacts = [
    explainability_output_directory
    / "permutation_importance.csv",

    explainability_output_directory
    / "shap_encoded_global_importance.csv",

    explainability_output_directory
    / "shap_original_feature_importance.csv",

    explainability_output_directory
    / "customer_reason_codes.csv",

    explainability_output_directory
    / "customer_reason_summary.csv",

    explainability_output_directory
    / "fairness_group_metrics.csv",

    explainability_output_directory
    / "explained_customer_predictions.csv",

    explainability_output_directory
    / "explainability_metadata.json",

    figures_directory
    / "permutation_importance.png",

    figures_directory
    / "shap_encoded_global_bar.png",

    figures_directory
    / "shap_encoded_beeswarm.png",

    figures_directory
    / "shap_original_feature_importance.png",

    figures_directory
    / "highest_risk_customer_waterfall.png",

    figures_directory
    / "highest_risk_original_feature_drivers.png"
]


artifact_check = pd.DataFrame(
    [
        {
            "artifact": path.name,

            "location": str(
                path.parent
            ),

            "exists": path.exists()
        }

        for path in (
            required_explainability_artifacts
        )
    ]
)


display(artifact_check)


missing_artifacts = artifact_check[
    artifact_check["exists"] == False
]


if missing_artifacts.empty:

    print(
        "\nAll required Notebook 6 "
        "artifacts were created."
    )

else:

    print(
        "\nMissing artifacts:"
    )

    display(
        missing_artifacts
    )

In [ ]:
print("=" * 65)

print(
    "NOTEBOOK 6 — MODEL EXPLAINABILITY COMPLETE"
)

print("=" * 65)

print(
    f"Final model: "
    f"{selected_model_name}"
)

print(
    f"Probability source: "
    f"{probability_source}"
)

print(
    f"Selected threshold: "
    f"{selected_threshold:.3f}"
)

print(
    f"Customers explained: "
    f"{len(X_explain_raw)}"
)

print(
    f"Original features: "
    f"{len(original_feature_order)}"
)

print(
    f"Encoded features: "
    f"{len(readable_encoded_feature_names)}"
)

print(
    f"Top permutation feature: "
    f"{permutation_importance_df.loc[0, 'feature']}"
)

print(
    f"Top SHAP feature: "
    f"{raw_shap_global_importance.loc[0, 'feature']}"
)

print(
    "Maximum probability reconstruction error:",
    f"{maximum_probability_reconstruction_error:.8f}"
)

print(
    "\nPermutation importance explains:"
)

print(
    "The complete calibrated final pipeline"
)

print(
    "\nNative TreeSHAP explains:"
)

print(
    "The underlying XGBoost model "
    "in raw-margin/log-odds space"
)

print(
    "\nOutputs generated:"
)

print(
    "• Global permutation importance"
)

print(
    "• Global encoded-feature SHAP importance"
)

print(
    "• Global original-feature SHAP importance"
)

print(
    "• Individual customer explanations"
)

print(
    "• RAG-ready churn reason codes"
)

print(
    "• Gender and senior-citizen fairness audit"
)

print(
    "\nNext stage:"
)

print(
    "SQL database design and prediction loading"
)